In [ ]:
!pip install opencv-python  # Install OpenCV for image processing
!pip install numpy          # Install NumPy for numerical operations
!pip install pandas         # Install Pandas for data manipulation
!pip install matplotlib     # Install Matplotlib for data visualization
!pip install scikit-learn    # Install Scikit-learn for machine learning utilities
!pip install tqdm           # Install tqdm for progress bar visualization

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)


### Step 0: Data Integrity & De-duplication
In this initial stage of the Project Bariq pipeline, we focus on establishing a clean and reliable dataset. Medical imaging datasets often contain redundant or corrupted files which can lead to data leakage and biased evaluation metrics (specifically inflated AUC/Accuracy). This script automates the cleaning process by ensuring every image is unique and structurally sound.


In [ ]:
import os                                       # Import OS module for directory handling
import hashlib                                  # Import Hashlib for file hashing
import shutil                                   # Import Shutil for file operations
import cv2                                      # Import OpenCV for image processing

# --- Path Configuration
BASE_DIR = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set" # Define main project path

RAW_GLAUCOMA = os.path.join(BASE_DIR, "medical", "all Glaucoma")      # Source path for glaucoma images
RAW_NORMAL   = os.path.join(BASE_DIR, "medical", "all normal")        # Source path for normal images

PROCESSED_BASE = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")    # Define output base directory
OUT_GLAUCOMA   = os.path.join(PROCESSED_BASE, "Step0_Clean_Data", "Glaucoma") # Clean glaucoma output path
OUT_NORMAL     = os.path.join(PROCESSED_BASE, "Step0_Clean_Data", "Normal")   # Clean normal output path

os.makedirs(OUT_GLAUCOMA, exist_ok=True)        # Create glaucoma directory if missing
os.makedirs(OUT_NORMAL,   exist_ok=True)        # Create normal directory if missing

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff") # Define allowed image extensions

# --- Core Hashing Logic
def get_md5(filepath):                          # Function to generate MD5 hash
    with open(filepath, "rb") as f:             # Open file in binary mode
        return hashlib.md5(f.read()).hexdigest() # Compute and return hex digest

# --- Data Cleaning Process
def remove_duplicates(input_dir, output_dir, label): # Main cleaning function
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)] # List valid images

    seen_hashes = {}                            # Dictionary for unique hash tracking
    duplicates  = []                            # List to store duplicate names
    unique      = 0                             # Counter for unique files

    for fname in files:                         # Iterate through each file
        fpath = os.path.join(input_dir, fname)  # Get full file path

        img = cv2.imread(fpath)                 # Load image for integrity check
        if img is None:                         # If file is corrupted or unreadable
            print(f" [SKIP] Corrupted file: {fname}") # Log skipping corrupted file
            continue                            # Proceed to next file

        file_hash = get_md5(fpath)              # Generate hash for uniqueness check

        if file_hash in seen_hashes:            # If hash already exists
            duplicates.append((fname, seen_hashes[file_hash])) # Track as duplicate
        else:                                   # If new unique hash
            seen_hashes[file_hash] = fname      # Store hash and filename
            out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png") # Define output path
            shutil.copy2(fpath, out_path)       # Copy file with metadata
            unique += 1                         # Increment unique counter

    print(f"\n[{label}]")                       # Print class label
    print(f"  Total scanned  : {len(files)}")   # Display total files found
    print(f"  Duplicates     : {len(duplicates)}") # Display duplicate count
    print(f"  Unique saved   : {unique}")       # Display unique files saved

    if duplicates:                              # If duplicates were found
        print("  Sample Duplicate Pairs:")       # Log duplicates header
        for dup, orig in duplicates[:10]:       # Loop through first 10 samples
            print(f"    {dup}  Matches  {orig}") # Print duplicate relationship
        if len(duplicates) > 10:                # If more than 10 duplicates
            print(f"    ... and {len(duplicates)-10} more items.") # Show remaining count

    return len(files), len(duplicates), unique  # Return processing statistics

# --- Main Execution Loop
total_g, dup_g, uniq_g = remove_duplicates(RAW_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)") # Process glaucoma
total_n, dup_n, uniq_n = remove_duplicates(RAW_NORMAL,   OUT_NORMAL,   "Normal (NRG)") # Process normal

print("\n" + "="*50)                            # Print separator
print(f" Step 0: Pre-processing Completed")     # Print completion status
print(f" Glaucoma Class : {total_g} total -> {uniq_g} unique ({dup_g} removed)") # Glaucoma summary
print(f" Normal Class   : {total_n} total -> {uniq_n} unique ({dup_n} removed)") # Normal summary
print(f" Storage Path   : Step0_Clean_Data/")   # Output location info
print("="*50)                                   # Print bottom border


[Glaucoma (RG)]
  Total scanned  : 4770
  Duplicates     : 0
  Unique saved   : 4770

[Normal (NRG)]
  Total scanned  : 4770
  Duplicates     : 0
  Unique saved   : 4770

 Step 0: Pre-processing Completed
 Glaucoma Class : 4770 total -> 4770 unique (0 removed)
 Normal Class   : 4770 total -> 4770 unique (0 removed)
 Storage Path   : Step0_Clean_Data/



### Step 1: Image Resizing Pipeline for Project BARIQ
------------------------------------------------
Purpose: Standardizes all fundus images to 224x224 pixels.

Input: Cleaned unique images from 'Step0_Clean_Data'.

Output: Resized PNG images for MobileNetV3 compatibility.



In [ ]:

import os
import cv2

#  Path Configuration (Linked to Step 0 Output)
# The base directory where the processed project data lives
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"

# Input paths: Reading directly from the output of the Cleaning stage (Step 0)
INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "Step0_Clean_Data", "Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "Step0_Clean_Data", "Normal")

# Output paths: Creating the Resizing stage in the pipeline
OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Step1_Resized", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Step1_Resized", "Normal")

# Ensure the new directory structure exists
os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

# Processing Constants
SUPPORTED       = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
TARGET_SIZE     = (224, 224)  # Standard input resolution for MobileNetV3

#  Core Resizing Logic (RGB Preserved)
def resize_rgb_images(input_path, output_path, label):
    """
    Iterates through the directory, maintains 3-channel (RGB) integrity,
    and applies INTER_AREA for optimal downsampling quality.
    """
    # Fetch all valid image files
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Standardizing {label} to {TARGET_SIZE[0]}x{TARGET_SIZE[1]} RGB...]")

    for fname in files:
        full_input_path = os.path.join(input_path, fname)

        # Load image (OpenCV loads as BGR with 3 channels by default)
        img = cv2.imread(full_input_path)

        # Safety check for corrupted files
        if img is None:
            print(f"  [WARNING] Integrity error on: {fname}")
            skipped_count += 1
            continue

        # Resize using INTER_AREA: Best for shrinking images without losing vital medical details
        resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

        # Save as PNG to avoid lossy compression artifacts
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), resized)
        processed_count += 1

    return len(files), processed_count

#  Execution Entry Point
if __name__ == "__main__":
    # Process the Glaucoma class
    total_g, saved_g = resize_rgb_images(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Process the Normal class
    total_n, saved_n = resize_rgb_images(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    # Final Execution Summary
    print("\n" + "="*60)
    print(f" STEP 1: RGB RESIZING COMPLETED")
    print(f" Configuration : {TARGET_SIZE[0]}x{TARGET_SIZE[1]} | 3 Channels (BGR/RGB)")
    print(f" Glaucoma Class: {saved_g}/{total_g} images standardized.")
    print(f" Normal Class  : {saved_n}/{total_n} images standardized.")
    print(f" Destination   : {os.path.join(BASE_DIR, 'Step1_Resized')}")
    print("="*60)


[Standardizing Glaucoma (RG) to 224x224 RGB...]

[Standardizing Normal (NRG) to 224x224 RGB...]

 STEP 1: RGB RESIZING COMPLETED
 Configuration : 224x224 | 3 Channels (BGR/RGB)
 Glaucoma Class: 4770/4770 images standardized.
 Normal Class  : 4770/4770 images standardized.
 Destination   : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step1_Resized



### Step 2: Image Quality Gate Pipeline for Project BARIQ
---------------------------------------------------
Purpose: Filters out low-quality images (Blurry, Dark, or Overexposed).

Why this step?: High-quality inputs ensure reliable feature extraction and
                 accurate Grad-CAM heatmaps for medical diagnosis.

Input: Standardized RGB images from 'Step1_Resized'.

Output: High-quality images in 'Step2_QualityGate' and discarded ones in '_Rejected'.


In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Step 1 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"

# Input paths: Reading from the Resizing stage
INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "Step1_Resized", "Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "Step1_Resized", "Normal")

# Output paths: Creating the Quality Gate stage
OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Step2_QualityGate", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Step2_QualityGate", "Normal")

# Directory for rejected images for auditing purposes
REJECTED_DIR    = os.path.join(BASE_DIR, "Step2_QualityGate", "_Rejected")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)
os.makedirs(REJECTED_DIR, exist_ok=True)

# Processing Constants & Thresholds
SUPPORTED        = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
BLUR_THRESHOLD   = 50.0    # Variance of Laplacian (Higher = Sharper)
DARK_THRESHOLD   = 20.0    # Mean pixel intensity (Below this is underexposed)
BRIGHT_THRESHOLD = 235.0   # Mean pixel intensity (Above this is overexposed)

#  Core Quality Filtering Logic
def run_quality_check(input_path, output_path, label):
    """
    Evaluates image sharpness and exposure levels.
    Only images meeting clinical quality standards are passed to the next stage.
    """
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    passed_count   = 0
    rejected_list  = []

    print(f"\n[Filtering {label} based on Medical Quality Standards...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            rejected_list.append((fname, "Corrupted"))
            continue

        # Convert to Grayscale for thresholding analysis
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 1. Sharpness Check: Compute Laplacian variance
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

        # 2. Exposure Check: Compute mean brightness
        brightness = gray.mean()

        # Decision Logic
        if blur_score < BLUR_THRESHOLD:
            reason = f"Blurry (Score: {blur_score:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"BLUR_{fname}"), img)

        elif brightness < DARK_THRESHOLD:
            reason = f"Underexposed (Mean: {brightness:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"DARK_{fname}"), img)

        elif brightness > BRIGHT_THRESHOLD:
            reason = f"Overexposed (Mean: {brightness:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"BRIGHT_{fname}"), img)

        else:
            # Image Passed: Save as PNG to the QualityGate folder
            save_path = os.path.join(output_path, os.path.splitext(fname)[0] + ".png")
            cv2.imwrite(save_path, img)
            passed_count += 1

    # Detailed Summary for this class
    print(f"  Total Processed: {len(files)}")
    print(f"  Successfully Passed: {passed_count}")
    print(f"  Rejected for Quality: {len(rejected_list)}")

    return len(files), passed_count

#  Execution Entry Point
if __name__ == "__main__":
    # Process the Glaucoma class
    total_g, pass_g = run_quality_check(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Process the Normal class
    total_n, pass_n = run_quality_check(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    # Final Summary Report
    print("\n" + "="*60)
    print(f" STEP 2: QUALITY GATE COMPLETED")
    print(f" Metrics       : Sharpness Var > {BLUR_THRESHOLD} | Brightness {DARK_THRESHOLD}-{BRIGHT_THRESHOLD}")
    print(f" Glaucoma Class: {pass_g}/{total_g} images passed.")
    print(f" Normal Class  : {pass_n}/{total_n} images passed.")
    print(f" Audit Path    : {REJECTED_DIR}")
    print("="*60)


[Filtering Glaucoma (RG) based on Medical Quality Standards...]
  Total Processed: 4770
  Successfully Passed: 4179
  Rejected for Quality: 591

[Filtering Normal (NRG) based on Medical Quality Standards...]
  Total Processed: 4770
  Successfully Passed: 4323
  Rejected for Quality: 447

 STEP 2: QUALITY GATE COMPLETED
 Metrics       : Sharpness Var > 50.0 | Brightness 20.0-235.0
 Glaucoma Class: 4179/4770 images passed.
 Normal Class  : 4323/4770 images passed.
 Audit Path    : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step2_QualityGate\_Rejected



### Step 3: LAB-CLAHE Contrast Enhancement Pipeline for Project BARIQ
---------------------------------------------------------------
Purpose: Enhances local contrast while strictly preserving clinical color integrity.

Technique: Converts BGR to LAB color space, applies CLAHE to the L-channel (luminance),
 and merges back. This prevents color shifting during enhancement.

Input: High-quality RGB images from 'Step2_QualityGate'.

Output: Enhanced color images in 'Step3_CLAHE'.


In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Step 2 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"

# Input paths: Reading from the Quality Gate stage (Step 2)
INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "Step2_QualityGate", "Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "Step2_QualityGate", "Normal")

# Output paths: Creating the CLAHE stage in the pipeline
OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Step3_CLAHE", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Step3_CLAHE", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

# Processing Constants (Medical standards for retinal imaging)
SUPPORTED       = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
CLIP_LIMIT      = 2.0   # Threshold for contrast limiting to avoid noise
TILE_GRID       = (8, 8) # Neighborhood size for adaptive equalization

# Core LAB-CLAHE Logic
def apply_medical_lab_clahe(input_path, output_path, label):
    """
    Applies CLAHE specifically to the Lightness channel of the LAB color space.
    This maintains the diagnostic color information while revealing hidden disc details.
    """
    # Initialize the CLAHE algorithm
    clahe_tool = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=TILE_GRID)

    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Applying LAB-CLAHE Enhancement for {label}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            print(f"  [WARNING] Could not read file: {fname}")
            skipped_count += 1
            continue

        # 1. Convert BGR to LAB color space
        lab_img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

        # 2. Split into L, A, and B channels
        l_chan, a_chan, b_chan = cv2.split(lab_img)

        # 3. Apply CLAHE only to the L-channel (Luminance)
        l_enhanced = clahe_tool.apply(l_chan)

        # 4. Merge the enhanced L-channel back with original A and B channels
        merged_lab = cv2.merge([l_enhanced, a_chan, b_chan])

        # 5. Convert back to BGR for standard storage/display
        final_bgr = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

        # Save results as PNG
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), final_bgr)
        processed_count += 1

    return len(files), processed_count

# Execution Entry Point
if __name__ == "__main__":
    # Process the Glaucoma class
    total_g, saved_g = apply_medical_lab_clahe(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Process the Normal class
    total_n, saved_n = apply_medical_lab_clahe(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    # Final Execution Summary
    print("\n" + "="*60)
    print(f" STEP 3: LAB-CLAHE ENHANCEMENT COMPLETED")
    print(f" Parameters    : ClipLimit={CLIP_LIMIT}, TileGrid={TILE_GRID}")
    print(f" Strategy      : Lightness-only contrast (A/B channels preserved)")
    print(f" Glaucoma Class: {saved_g}/{total_g} images enhanced.")
    print(f" Normal Class  : {saved_n}/{total_n} images enhanced.")
    print(f" Saved in      : {os.path.join(BASE_DIR, 'Step3_CLAHE')}")
    print("="*60)


[Applying LAB-CLAHE Enhancement for Glaucoma (RG)...]

[Applying LAB-CLAHE Enhancement for Normal (NRG)...]

 STEP 3: LAB-CLAHE ENHANCEMENT COMPLETED
 Parameters    : ClipLimit=2.0, TileGrid=(8, 8)
 Strategy      : Lightness-only contrast (A/B channels preserved)
 Glaucoma Class: 4179/4179 images enhanced.
 Normal Class  : 4323/4323 images enhanced.
 Saved in      : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step3_CLAHE



### Step 4: Circular Masking Pipeline for Project BARIQ
--------------------------------------------------
Purpose: Applies a circular mask to isolate the fundus (retina) area.

Why this step?: Standardizes the Region of Interest (ROI) and eliminates
                 irrelevant corner artifacts that may interfere with training.

Input: Enhanced RGB images from 'Step3_CLAHE'.

Output: Masked RGB images stored in 'Step4_CircularMask'.


In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Step 3 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"

# Input paths: Reading from the LAB-CLAHE stage (Step 3)
INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "Step3_CLAHE", "Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "Step3_CLAHE", "Normal")

# Output paths: Creating the Masking stage in the pipeline
OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Step4_CircularMask", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Step4_CircularMask", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

# Processing Constants
SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Core Masking Logic
def apply_retinal_mask(input_path, output_path, label):
    """
    Generates a binary circular mask and performs a bitwise-AND operation
    with the source image to black out the corners.
    """
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Applying Circular Masking for {label}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            print(f"  [WARNING] File error: {fname}")
            skipped_count += 1
            continue

        # Get image dimensions (Assumed 224x224 from Step 1)
        h, w   = img.shape[:2]
        center = (w // 2, h // 2)
        radius = min(w, h) // 2

        # 1. Create an empty black mask
        mask = np.zeros((h, w), dtype=np.uint8)

        # 2. Draw a filled white circle (255) on the mask
        cv2.circle(mask, center, radius, 255, thickness=-1)

        # 3. Apply mask to the image (Bitwise-AND across all 3 channels)
        # Only pixels inside the white circle are kept; others become black (0)
        masked_img = cv2.bitwise_and(img, img, mask=mask)

        # Save results
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), masked_img)
        processed_count += 1

    return len(files), processed_count

# Execution Entry Point
if __name__ == "__main__":
    # Process the Glaucoma class
    total_g, saved_g = apply_retinal_mask(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Process the Normal class
    total_n, saved_n = apply_retinal_mask(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    # Final Execution Summary
    print("\n" + "="*60)
    print(f" STEP 4: CIRCULAR MASKING COMPLETED")
    print(f" ROI Target    : Centered Circle (Radius = Height/2)")
    print(f" Glaucoma Class: {saved_g}/{total_g} images masked.")
    print(f" Normal Class  : {saved_n}/{total_n} images masked.")
    print(f" Saved in      : {os.path.join(BASE_DIR, 'Step4_CircularMask')}")
    print("="*60)


[Applying Circular Masking for Glaucoma (RG)...]

[Applying Circular Masking for Normal (NRG)...]

 STEP 4: CIRCULAR MASKING COMPLETED
 ROI Target    : Centered Circle (Radius = Height/2)
 Glaucoma Class: 4179/4179 images masked.
 Normal Class  : 4323/4323 images masked.
 Saved in      : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step4_CircularMask



### Step 5: RGB Pixel Normalization Pipeline for Project BARIQ
---------------------------------------------------------
Purpose: Standardizes pixel intensity values across all 3 RGB channels.

Why Normalize?: Scaling values to [0, 1] prevents numerical instability
                 and ensures efficient gradient flow during backpropagation.

Input: Masked RGB images from 'Step4_CircularMask'.

Output: Standardized RGB images stored in 'Step5_Normalized'.


In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Step 4 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"


INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "Step4_CircularMask", "Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "Step4_CircularMask", "Normal")


OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Step5_Normalized", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Step5_Normalized", "Normal")
os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

# Processing Constants
SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Core Normalization Logic
def normalize_rgb_data(input_path, output_path, label):
    """
    Standardizes RGB intensity. Note: Images are saved as uint8 for storage
    visibility, but the pipeline logic confirms data readiness for
    rescale=1./255 during model training.
    """
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Normalizing 3-Channel RGB Values for {label}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path) # BGR format

        if img is None:
            print(f"  [WARNING] Could not access: {fname}")
            skipped_count += 1
            continue

        # 1. Convert to float32 for mathematical precision
        # 2. Rescale to [0.0, 1.0] range
        normalized_img = img.astype(np.float32) / 255.0

        # For storage purposes and folder auditing, we revert to uint8.
        # The actual floating-point rescale is reapplied in the training script.
        storage_img = (normalized_img * 255).astype(np.uint8)

        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), storage_img)
        processed_count += 1

    return len(files), processed_count

#  Execution Entry Point
if __name__ == "__main__":
    # Process the Glaucoma class
    total_g, saved_g = normalize_rgb_data(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Process the Normal class
    total_n, saved_n = normalize_rgb_data(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    # Final Execution Summary
    print("\n" + "="*60)
    print(f" STEP 5: RGB NORMALIZATION COMPLETED")
    print(f" Data Format   : Float32 Standardized (stored as 8-bit PNG)")
    print(f" Glaucoma Class: {saved_g}/{total_g} images normalized.")
    print(f" Normal Class  : {saved_n}/{total_n} images normalized.")
    print(f" Final Path    : {os.path.join(BASE_DIR, 'Step5_Normalized')}")
    print("="*60)


[Normalizing 3-Channel RGB Values for Glaucoma (RG)...]

[Normalizing 3-Channel RGB Values for Normal (NRG)...]

 STEP 5: RGB NORMALIZATION COMPLETED
 Data Format   : Float32 Standardized (stored as 8-bit PNG)
 Glaucoma Class: 4179/4179 images normalized.
 Normal Class  : 4323/4323 images normalized.
 Final Path    : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step5_Normalized


### Step 6: Final Dataset Partitioning for Project BARIQ
---------------------------------------------------
Purpose: Stratifies the processed RGB images into Training, Validation, and Testing sets.

Ratio: 70% Training (Model weights tuning), 15% Validation (Hyperparameter tuning),
       15% Testing (Final unseen performance evaluation).

Input: Final processed RGB images from 'Step5_Normalized'.

Output: Final directory structure in 'Step6_Split'.


In [ ]:

import os
import shutil
import random

# Path Configuration (Linked to Step 5 Output)
BASE_DIR    = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed"

INPUT_DIR   = os.path.join(BASE_DIR, "Step5_Normalized")

OUTPUT_DIR  = os.path.join(BASE_DIR, "Step6_Split")

CLASSES     = ["Glaucoma", "Normal"]
SPLITS      = {"Train": 0.70, "Val": 0.15, "Test": 0.15}
SUPPORTED   = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
SEED        = 42  # Ensures identical splitting results across different machines

# Directory Initialization
# Constructing: Step6_Split/{Train,Val,Test}/{Glaucoma,Normal}
for split in SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)

# Core Splitting Logic
def execute_dataset_partition(cls_name):
    """
    Randomly assigns images to folders while maintaining class labels.
    Uses shutil.copy2 to preserve file metadata.
    """
    source_path = os.path.join(INPUT_DIR, cls_name)
    files = [f for f in os.listdir(source_path) if f.lower().endswith(SUPPORTED)]

    # Deterministic shuffling for reproducibility
    random.seed(SEED)
    random.shuffle(files)

    total_count = len(files)
    n_train = int(total_count * SPLITS["Train"])
    n_val   = int(total_count * SPLITS["Val"])
    # Remainder logic to prevent loss of files due to decimal rounding
    n_test  = total_count - n_train - n_val

    # Mapping files to distribution buckets
    distribution = {
        "Train" : files[:n_train],
        "Val"   : files[n_train : n_train + n_val],
        "Test"  : files[n_train + n_val:]
    }

    # Execute file movement (copying to keep Step5 as a backup)
    for split_key, file_list in distribution.items():
        for filename in file_list:
            src_file = os.path.join(source_path, filename)
            dst_file = os.path.join(OUTPUT_DIR, split_key, cls_name, filename)
            shutil.copy2(src_file, dst_file)

    print(f"  [STAGING {cls_name:8}] Total: {total_count:4} | Train: {n_train:4} | Val: {n_val:3} | Test: {n_test:3}")

# Execution Entry Point
if __name__ == "__main__":
    print(f"Finalizing BARIQ Dataset Split (Seed: {SEED})...")

    for cls in CLASSES:
        execute_dataset_partition(cls)

    print("\n" + "="*60)
    print(f" STEP 6: DATASET SPLIT COMPLETED SUCCESSFULLY")
    print(f" Distribution  : 70% Train | 15% Val | 15% Test")
    print(f" Consistency   : Randomized seed {SEED} applied.")
    print(f" Target Folder : {OUTPUT_DIR}")
    print("="*60)

Finalizing BARIQ Dataset Split (Seed: 42)...
  [STAGING Glaucoma] Total: 4179 | Train: 2925 | Val: 626 | Test: 628
  [STAGING Normal  ] Total: 4323 | Train: 3026 | Val: 648 | Test: 649

 STEP 6: DATASET SPLIT COMPLETED SUCCESSFULLY
 Distribution  : 70% Train | 15% Val | 15% Test
 Consistency   : Randomized seed 42 applied.
 Target Folder : C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Step6_Split
